# `Dynamic AI Knowledge Assistant`

In [63]:
# ============================================================
# PHASE 4 – TEXT CHUNKING
# STEP 1 – IMPORTS
# ============================================================

from pathlib import Path
import re
import time
import hashlib

import pandas as pd
import numpy as np

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")

Libraries imported successfully.


In [64]:
# ============================================================
# STEP 2 – PROJECT PATHS
# ============================================================

project_root = None

for parent in [Path.cwd()] + list(Path.cwd().parents):

    candidate = (parent/ "data"/ "processed"/ "cleaned")
    if candidate.exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError("Project root could not be detected." )

CLEANED_DIR = (project_root/ "data" / "processed"/ "cleaned")
METADATA_PATH = (project_root/ "data"/ "metadata"/ "metadata.csv")
CHUNKS_DIR = ( project_root / "data"/ "processed"/ "chunks")
CHUNKS_DIR.mkdir(parents=True,exist_ok=True)

print("Project root :", project_root)
print("Cleaned data :", CLEANED_DIR)
print("Metadata     :", METADATA_PATH)
print("Chunks output:", CHUNKS_DIR)

Project root : c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant
Cleaned data : c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\cleaned
Metadata     : c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\metadata\metadata.csv
Chunks output: c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\chunks


In [65]:
# ============================================================
# STEP 3 – LOAD CLEANED CORPUS
# ============================================================

cleaned_files = sorted(
    CLEANED_DIR.glob("*.txt")
)

metadata_df = pd.read_csv(
    METADATA_PATH
)

print("Cleaned documents :", len(cleaned_files))
print("Metadata records  :", len(metadata_df))

assert len(cleaned_files) == 128
assert len(metadata_df) == 128

print("\nPhase 3 corpus verification PASSED.")

Cleaned documents : 128
Metadata records  : 128

Phase 3 corpus verification PASSED.


In [66]:
# ============================================================
# STEP 4 – LOAD DOCUMENTS
# ============================================================
metadata_source = pd.read_csv(
    project_root
    / "data"
    / "metadata"
    / "metadata_phase1_backup.csv"
)

metadata_lookup = (
    metadata_source
    .set_index("file_name")
    .to_dict("index")
)

documents = []

for file_path in cleaned_files:

    if file_path.name not in metadata_lookup:
        raise ValueError(
            f"No metadata found for {file_path.name}"
        )

    text = file_path.read_text(
        encoding="utf-8"
    )

    documents.append({
        "file_name": file_path.name,
        "text": text,
        "article_name": metadata_lookup[
            file_path.name
        ]["title"],
        "url": metadata_lookup[
            file_path.name
        ]["url"],
        "category": metadata_lookup[
            file_path.name
        ]["category"]
    })

print(
    "Documents loaded:",
    len(documents)
)

Documents loaded: 128


In [67]:
# ============================================================
# STEP 5 – COMMON CHUNKING CONFIGURATION
# ============================================================

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

print("Chunk size    :", CHUNK_SIZE)
print("Chunk overlap :", CHUNK_OVERLAP)

# ============================================================
# STEP 6 – CHUNK ID
# ============================================================

def create_document_id(file_name):
    return hashlib.sha1(
        file_name.encode("utf-8")
    ).hexdigest()[:12]


def create_chunk_id(
    document_id,
    chunk_index
):
    return (
        f"{document_id}_"
        f"chunk_{chunk_index:04d}"
    )


# ============================================================
# STEP 7 – COMMON CHUNK RECORD
# ============================================================

def create_chunk_record(
    document,
    chunk_text,
    chunk_index,
    method
):

    document_id = create_document_id(
        document["file_name"]
    )

    return {
        "chunk_id": create_chunk_id(
            document_id,
            chunk_index
        ),
        "document_id": document_id,
        "article_name": document[
            "article_name"
        ],
        "url": document["url"],
        "category": document["category"],
        "source_file": document[
            "file_name"
        ],
        "chunk_index": chunk_index,
        "chunking_method": method,
        "chunk_text": chunk_text,
        "chunk_length": len(chunk_text)
    }

Chunk size    : 1000
Chunk overlap : 200


In [68]:
# ============================================================
# STEP 8 – FIXED CHUNKING
# ============================================================

def fixed_chunk_text(
    text,
    chunk_size=1000,
    chunk_overlap=200
):

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must be smaller "
            "than chunk_size."
        )

    chunks = []

    start = 0
    text_length = len(text)

    step = (
        chunk_size
        - chunk_overlap
    )

    while start < text_length:

        end = min(
            start + chunk_size,
            text_length
        )

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += step

    return chunks


# ============================================================
# TEST FIXED CHUNKING
# ============================================================

sample_document = documents[0]

fixed_test_chunks = fixed_chunk_text(
    sample_document["text"],
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

print(
    "Article:",
    sample_document["article_name"]
)

print(
    "Original characters:",
    len(sample_document["text"])
)

print(
    "Fixed chunks:",
    len(fixed_test_chunks)
)

print(
    "First chunk length:",
    len(fixed_test_chunks[0])
)

print("\nFirst chunk:")
print(fixed_test_chunks[0][:1000])

Article: 1.58-bit large language model
Original characters: 4272
Fixed chunks: 6
First chunk length: 1000

First chunk:
A 1.58-bit large language model (also known as a ternary LLM) is a type of large language model (LLM) designed to be computationally efficient. It achieves this by using weights that are restricted to only three values: −1, 0, and +1. This restriction significantly reduces the model's memory footprint and allows for faster processing, as computationally expensive multiplication operations can be replaced with lower-cost additions. This contrasts with traditional models that use 16-bit floating-point numbers (FP16 or BF16) for their weights.
Studies have shown that for models up to several billion parameters, the performance of 1.58-bit LLMs on various tasks is comparable to their full-precision counterparts. This approach could enable powerful AI to run on less specialized and lower-power hardware.
The name "1.58-bit" comes from the fact that a value with three states

In [69]:
# ============================================================
# FIXED CHUNKING VALIDATION
# ============================================================

assert len(fixed_test_chunks) > 0

assert all(
    len(chunk) <= CHUNK_SIZE
    for chunk in fixed_test_chunks
)

assert all(
    chunk.strip()
    for chunk in fixed_test_chunks
)

print(
    "Fixed chunking sample validation PASSED."
)

Fixed chunking sample validation PASSED.


In [70]:
# ============================================================
# STEP 9 – RECURSIVE CHUNKING
# ============================================================

recursive_splitter = (
    RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=[
            "\n\n",
            "\n",
            ". ",
            "? ",
            "! ",
            " ",
            ""
        ],
        length_function=len
    )
)

def recursive_chunk_text(text):

    return recursive_splitter.split_text(
        text
    )

# ============================================================
# TEST RECURSIVE CHUNKING
# ============================================================

recursive_test_chunks = recursive_chunk_text(
    sample_document["text"]
)

print(
    "Recursive chunks:",
    len(recursive_test_chunks)
)

print(
    "First chunk length:",
    len(recursive_test_chunks[0])
)

print("\nFirst recursive chunk:")
print(
    recursive_test_chunks[0][:1000]
)

Recursive chunks: 6
First chunk length: 952

First recursive chunk:
A 1.58-bit large language model (also known as a ternary LLM) is a type of large language model (LLM) designed to be computationally efficient. It achieves this by using weights that are restricted to only three values: −1, 0, and +1. This restriction significantly reduces the model's memory footprint and allows for faster processing, as computationally expensive multiplication operations can be replaced with lower-cost additions. This contrasts with traditional models that use 16-bit floating-point numbers (FP16 or BF16) for their weights.
Studies have shown that for models up to several billion parameters, the performance of 1.58-bit LLMs on various tasks is comparable to their full-precision counterparts. This approach could enable powerful AI to run on less specialized and lower-power hardware.
The name "1.58-bit" comes from the fact that a value with three states contains


log

2


⁡
3
≈
1.58


{\displaystyle \lo

In [71]:
# ============================================================
# RECURSIVE CHUNKING VALIDATION
# ============================================================

assert len(recursive_test_chunks) > 0

assert all(
    len(chunk) <= CHUNK_SIZE
    for chunk in recursive_test_chunks
)

assert all(
    chunk.strip()
    for chunk in recursive_test_chunks
)

print(
    "Recursive chunking sample validation PASSED."
)

Recursive chunking sample validation PASSED.


In [72]:
# ============================================================
# STEP 10 – SEMANTIC CHUNKING
# Load embedding model for semantic chunking
# ============================================================

semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Semantic embedding model loaded successfully.")

# ============================================================
# SEMANTIC CHUNKING – WIKIPEDIA-AWARE UNIT SPLITTING
# ============================================================

def split_into_semantic_units(text):
    """
    Split cleaned Wikipedia text into semantic units.

    Rules:
    - Preserve Wikipedia section headings.
    - Preserve mathematical expressions.
    - Join continuation lines belonging to the same sentence.
    - Avoid sentence breaks after decimal numbers.
    """

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # --------------------------------------------------------
    # Normalize spaces/tabs inside individual lines
    # --------------------------------------------------------

    lines = []

    for line in text.split("\n"):

        line = re.sub(
            r"[ \t]+",
            " ",
            line
        ).strip()

        if line:
            lines.append(line)

    # --------------------------------------------------------
    # Reconstruct logical blocks
    #
    # Wikipedia headings remain independent.
    # Other consecutive lines are joined so mathematical
    # expressions split across lines remain intact.
    # --------------------------------------------------------

    blocks = []
    current_block = []

    for line in lines:

        # Wikipedia section heading
        if re.fullmatch(
            r"==+\s*[^=\n]+?\s*==+",
            line
        ):

            if current_block:
                blocks.append(
                    " ".join(current_block)
                )
                current_block = []

            blocks.append(line)

        else:
            current_block.append(line)

    if current_block:
        blocks.append(
            " ".join(current_block)
        )

    # --------------------------------------------------------
    # Sentence segmentation
    # --------------------------------------------------------

    units = []

    for block in blocks:

        block = block.strip()

        if not block:
            continue

        # Preserve section headings
        if re.fullmatch(
            r"==+\s*[^=\n]+?\s*==+",
            block
        ):
            units.append(block)
            continue

        # Split only at genuine sentence-ending punctuation.
        #
        # Do not split after decimal numbers such as 1.58.
        #
        sentences = re.split(
            r"(?<!\d)(?<=[.!?])\s+",
            block
        )

        for sentence in sentences:

            sentence = sentence.strip()

            if sentence:
                units.append(sentence)

    return units

semantic_units = split_into_semantic_units(
    sample_document["text"]
)

print(
    "Total semantic units:",
    len(semantic_units)
)

print("\nDetected Wikipedia headings:\n")

for i, unit in enumerate(semantic_units):

    if re.fullmatch(
        r"==+\s*[^=\n]+?\s*==+",
        unit
    ):
        print(
            f"{i}: {unit}"
        )


# ============================================================
# WIKIPEDIA-AWARE SEMANTIC CHUNKING
# ============================================================

def is_section_heading(unit):
    """
    Return True when a text unit is a Wikipedia-style
    section heading.
    """

    return bool(
        re.fullmatch(
            r"==+\s*[^=\n]+?\s*==+",
            unit
        )
    )

def calculate_semantic_similarities(
    units,
    model
):

    if len(units) < 2:
        return []

    embeddings = model.encode(
        units,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    similarities = np.sum(
        embeddings[:-1] * embeddings[1:],
        axis=1
    )

    return similarities.tolist()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8166.76it/s]


Semantic embedding model loaded successfully.
Total semantic units: 56

Detected Wikipedia headings:

8: == BitNet ==
12: == Post-training quantization ==
16: == Critique ==
19: == References ==
20: == Sources ==


In [ ]:
# ============================================================
# SEMANTIC BOUNDARY DETECTION
# ============================================================

def find_semantic_boundaries(
    similarities,
    percentile=20
):

    if not similarities:
        return [], None

    threshold = float(
        np.percentile(
            similarities,
            percentile
        )
    )

    boundaries = [
        index
        for index, similarity
        in enumerate(similarities)
        if similarity <= threshold
    ]

    return boundaries, threshold

# ============================================================
# IMPROVED SEMANTIC CHUNKING
# ============================================================

def split_oversized_unit(
    text,
    max_chunk_size=1000
):
    """
    Split a semantic unit that exceeds the maximum chunk size.

    Sentence-aware first, then word-aware fallback.
    No text is discarded.
    """

    text = text.strip()

    if len(text) <= max_chunk_size:
        return [text]

    # --------------------------------------------------------
    # First try sentence-level splitting
    # --------------------------------------------------------

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    # --------------------------------------------------------
    # If sentence splitting gives useful pieces,
    # group them without exceeding the limit.
    # --------------------------------------------------------

    chunks = []
    current = ""

    for sentence in sentences:

        if not current:

            if len(sentence) <= max_chunk_size:
                current = sentence

            else:
                # Sentence itself is still too long.
                # Fall back to word-level splitting.
                word_parts = sentence.split()

                word_chunk = ""

                for word in word_parts:

                    proposed = (
                        word
                        if not word_chunk
                        else word_chunk + " " + word
                    )

                    if len(proposed) <= max_chunk_size:
                        word_chunk = proposed

                    else:

                        if word_chunk:
                            chunks.append(
                                word_chunk.strip()
                            )

                        word_chunk = word

                if word_chunk:
                    chunks.append(
                        word_chunk.strip()
                    )

        else:

            proposed = (
                current + " " + sentence
            )

            if len(proposed) <= max_chunk_size:

                current = proposed

            else:

                chunks.append(
                    current.strip()
                )

                current = sentence

                # Handle a sentence larger than max size
                if len(current) > max_chunk_size:

                    word_parts = current.split()

                    current = ""

                    for word in word_parts:

                        proposed_word = (
                            word
                            if not current
                            else current + " " + word
                        )

                        if len(proposed_word) <= max_chunk_size:

                            current = proposed_word

                        else:

                            if current:
                                chunks.append(
                                    current.strip()
                                )

                            current = word

    if current:
        chunks.append(
            current.strip()
        )

    # --------------------------------------------------------
    # Final safety check
    # --------------------------------------------------------

    final_chunks = []

    for chunk in chunks:

        if len(chunk) <= max_chunk_size:

            final_chunks.append(chunk)

        else:

            # Absolute fallback:
            # hard split only if necessary.
            for start in range(
                0,
                len(chunk),
                max_chunk_size
            ):

                piece = chunk[
                    start:start + max_chunk_size
                ].strip()

                if piece:
                    final_chunks.append(piece)

    return final_chunks


def semantic_chunk_text(
    text,
    model,
    percentile=20,
    max_chunk_size=1000
):

    units = split_into_semantic_units(text)

    if not units:
        return []

    # --------------------------------------------------------
    # Calculate semantic similarities
    # --------------------------------------------------------

    similarities = calculate_semantic_similarities(
        units,
        model
    )

    boundaries, threshold = (
        find_semantic_boundaries(
            similarities,
            percentile=percentile
        )
    )

    boundary_set = set(boundaries)

    chunks = []
    current_chunk = []

    # --------------------------------------------------------
    # Process semantic units
    # --------------------------------------------------------

    for index, unit in enumerate(units):

        unit = unit.strip()

        if not unit:
            continue

        # ----------------------------------------------------
        # Section heading
        #
        # Keep heading attached to following content.
        # ----------------------------------------------------

        if is_section_heading(unit):

            if current_chunk:

                chunk = " ".join(
                    current_chunk
                ).strip()

                if chunk:
                    chunks.append(chunk)

                current_chunk = []

            current_chunk.append(unit)

            continue

        # ----------------------------------------------------
        # If a single semantic unit is already too large,
        # split it safely.
        # ----------------------------------------------------

        if len(unit) > max_chunk_size:

            # Flush existing chunk first
            if current_chunk:

                chunk = " ".join(
                    current_chunk
                ).strip()

                if chunk:
                    chunks.append(chunk)

                current_chunk = []

            oversized_parts = split_oversized_unit(
                unit,
                max_chunk_size
            )

            chunks.extend(
                oversized_parts
            )

            continue

        # ----------------------------------------------------
        # Try adding the current unit
        # ----------------------------------------------------

        proposed_chunk = " ".join(
            current_chunk + [unit]
        ).strip()

        if (
            current_chunk
            and len(proposed_chunk) > max_chunk_size
        ):

            chunk = " ".join(
                current_chunk
            ).strip()

            if chunk:
                chunks.append(chunk)

            current_chunk = [unit]

        else:

            current_chunk.append(unit)

        # ----------------------------------------------------
        # Semantic boundary
        # ----------------------------------------------------

        if index in boundary_set:

            chunk = " ".join(
                current_chunk
            ).strip()

            if chunk:
                chunks.append(chunk)

            current_chunk = []

    # --------------------------------------------------------
    # Remaining content
    # --------------------------------------------------------

    if current_chunk:

        chunk = " ".join(
            current_chunk
        ).strip()

        if chunk:
            chunks.append(chunk)

    # --------------------------------------------------------
    # Final safety enforcement
    # --------------------------------------------------------

    final_chunks = []

    for chunk in chunks:

        if len(chunk) <= max_chunk_size:

            final_chunks.append(chunk)

        else:

            final_chunks.extend(
                split_oversized_unit(
                    chunk,
                    max_chunk_size
                )
            )

    return final_chunks

def postprocess_semantic_chunks(
    chunks,
    max_chunk_size=1000,
    min_chunk_size=50
):
    """
    Post-process semantic chunks without deleting
    meaningful content.

    Rules:
    1. Remove empty chunks.
    2. Merge heading-only chunks with following content.
    3. Merge short chunks with neighboring chunks when possible.
    4. Never exceed max_chunk_size.
    5. Never delete non-empty content.
    """

    # --------------------------------------------------------
    # Step 1: Remove empty chunks
    # --------------------------------------------------------

    chunks = [
        chunk.strip()
        for chunk in chunks
        if chunk.strip()
    ]

    if not chunks:
        return []

    # --------------------------------------------------------
    # Step 2: Merge heading-only chunks with following chunk
    # --------------------------------------------------------

    heading_processed = []

    i = 0

    while i < len(chunks):

        current = chunks[i]

        is_heading = bool(
            re.fullmatch(
                r"=+\s*[^=\n]+?\s*=+",
                current
            )
        )

        if (
            is_heading
            and i + 1 < len(chunks)
        ):

            merged = (
                current + " " + chunks[i + 1]
            ).strip()

            if len(merged) <= max_chunk_size:

                heading_processed.append(merged)
                i += 2
                continue

        heading_processed.append(current)
        i += 1

    # --------------------------------------------------------
    # Step 3: Merge short chunks with previous chunk
    # --------------------------------------------------------

    processed = []

    for chunk in heading_processed:

        if (
            len(chunk) < min_chunk_size
            and processed
        ):

            merged = (
                processed[-1] + " " + chunk
            ).strip()

            if len(merged) <= max_chunk_size:

                processed[-1] = merged
                continue

        processed.append(chunk)

    # --------------------------------------------------------
    # Step 4: Merge remaining short chunks with next chunk
    # --------------------------------------------------------

    final_chunks = []

    i = 0

    while i < len(processed):

        current = processed[i]

        if (
            len(current) < min_chunk_size
            and i + 1 < len(processed)
        ):

            merged = (
                current + " " + processed[i + 1]
            ).strip()

            if len(merged) <= max_chunk_size:

                final_chunks.append(merged)
                i += 2
                continue

        final_chunks.append(current)
        i += 1

    # --------------------------------------------------------
    # Step 5: Final heading-only safety check
    # --------------------------------------------------------

    cleaned_chunks = []

    i = 0

    while i < len(final_chunks):

        current = final_chunks[i]

        is_heading = bool(
            re.fullmatch(
                r"=+\s*[^=\n]+?\s*=+",
                current
            )
        )

        if (
            is_heading
            and i + 1 < len(final_chunks)
        ):

            merged = (
                current + " " + final_chunks[i + 1]
            ).strip()

            if len(merged) <= max_chunk_size:

                cleaned_chunks.append(merged)
                i += 2
                continue

        cleaned_chunks.append(current)
        i += 1

    return cleaned_chunks


# ============================================================
# RETEST SEMANTIC CHUNKING
# ============================================================

semantic_test_chunks = semantic_chunk_text(
    sample_document["text"],
    semantic_model,
    percentile=20,
    max_chunk_size=CHUNK_SIZE
)

print(
    "Article:",
    sample_document["article_name"]
)

print(
    "Original characters:",
    len(sample_document["text"])
)

print(
    "Semantic chunks:",
    len(semantic_test_chunks)
)

print(
    "Maximum chunk length:",
    max(
        len(chunk)
        for chunk in semantic_test_chunks
    )
)

# ============================================================
# SEMANTIC CHUNKING VALIDATION
# ============================================================

assert semantic_test_chunks

assert all(
    chunk.strip()
    for chunk in semantic_test_chunks
)

assert all(
    len(chunk) <= CHUNK_SIZE
    for chunk in semantic_test_chunks
)

print(
    "Semantic chunking sample validation PASSED."
)

for i, chunk in enumerate(
    semantic_test_chunks[:3]
):

    print("=" * 80)
    print(f"SEMANTIC CHUNK {i}")
    print(
        f"Length: {len(chunk)} characters"
    )
    print("=" * 80)
    print(chunk[:1500])
    print()


    semantic_test_chunks = semantic_chunk_text(
    sample_document["text"],
    semantic_model,
    percentile=20,
    max_chunk_size=CHUNK_SIZE
)

print(
    "Article:",
    sample_document["article_name"]
)

print(
    "Original characters:",
    len(sample_document["text"])
)

print(
    "Semantic chunks:",
    len(semantic_test_chunks)
)

print(
    "Maximum chunk length:",
    max(
        len(chunk)
        for chunk in semantic_test_chunks
    )
)

assert len(semantic_test_chunks) > 0

assert all(
    chunk.strip()
    for chunk in semantic_test_chunks
)

assert all(
    len(chunk) <= CHUNK_SIZE
    for chunk in semantic_test_chunks
)

print(
    "Semantic chunking sample validation PASSED."
)


for i, chunk in enumerate(
    semantic_test_chunks[:5]
):

    print("=" * 80)
    print(
        f"SEMANTIC CHUNK {i}"
    )
    print(
        f"Length: {len(chunk)} characters"
    )
    print("=" * 80)
    print(chunk[:1500])
    print()

Article: 1.58-bit large language model
Original characters: 4272
Semantic chunks: 14
Maximum chunk length: 809
Semantic chunking sample validation PASSED.
SEMANTIC CHUNK 0
Length: 809 characters
A 1.58-bit large language model (also known as a ternary LLM) is a type of large language model (LLM) designed to be computationally efficient. It achieves this by using weights that are restricted to only three values: −1, 0, and +1. This restriction significantly reduces the model's memory footprint and allows for faster processing, as computationally expensive multiplication operations can be replaced with lower-cost additions. This contrasts with traditional models that use 16-bit floating-point numbers (FP16 or BF16) for their weights. Studies have shown that for models up to several billion parameters, the performance of 1.58-bit LLMs on various tasks is comparable to their full-precision counterparts. This approach could enable powerful AI to run on less specialized and lower-power hardw

In [96]:
# =====================================================
# Create the output directory
# =====================================================


SEMANTIC_DIR = Path(
    "../data/processed/chunks/semantic"
)

SEMANTIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Semantic chunk directory:")
print(SEMANTIC_DIR.resolve())

# =====================================================
# Load All Documents
# =====================================================

cleaned_files = sorted(
    CLEANED_DIR.glob("*.txt")
)

print("Cleaned documents:", len(cleaned_files))

if len(cleaned_files) != 128:
    raise ValueError(
        f"Expected 128 cleaned documents, "
        f"found {len(cleaned_files)}."
    )

print("Cleaned corpus count validation PASSED.")

Semantic chunk directory:
C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\chunks\semantic
Cleaned documents: 128
Cleaned corpus count validation PASSED.


In [97]:
# =============================================
# Generate Semantic Chunks
# =============================================

semantic_summary = []

for file_path in cleaned_files:

    text = file_path.read_text(
        encoding="utf-8"
    ).strip()

    chunks = semantic_chunk_text(
        text,
        semantic_model,
        percentile=20,
        max_chunk_size=CHUNK_SIZE
    )

    chunks = postprocess_semantic_chunks(
    chunks,
    max_chunk_size=1000,
    min_chunk_size=50
)
    
    if not chunks:
        raise ValueError(
            f"No semantic chunks generated for "
            f"{file_path.name}"
        )

    # --------------------------------------------------------
    # Save chunks
    # --------------------------------------------------------

    output_path = (
        SEMANTIC_DIR /
        f"{file_path.stem}_semantic.txt"
    )

    output_text = "\n\n".join(
        chunks
    )

    output_path.write_text(
        output_text,
        encoding="utf-8"
    )

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    chunk_lengths = [
        len(chunk)
        for chunk in chunks
    ]

    semantic_summary.append({
        "file_name": file_path.name,
        "chunk_file": output_path.name,
        "original_characters": len(text),
        "chunk_count": len(chunks),
        "total_chunk_characters": sum(
            chunk_lengths
        ),
        "min_chunk_length": min(
            chunk_lengths
        ),
        "max_chunk_length": max(
            chunk_lengths
        ),
        "avg_chunk_length": sum(
            chunk_lengths
        ) / len(chunk_lengths)
    })

print(
    f"Semantic chunk generation completed for "
    f"{len(semantic_summary)} documents."
)


semantic_summary_df = pd.DataFrame(
    semantic_summary
)

semantic_summary_df.head()

Semantic chunk generation completed for 128 documents.


,file_name,chunk_file,original_characters,chunk_count,total_chunk_characters,min_chunk_length,max_chunk_length,avg_chunk_length
0,1.58-bit_large_language_model.txt,1.58-bit_large_language_model_semantic.txt,4272,12,4242,53,809,353.500000
1,A.I._Artificial_Intelligence.txt,A.I._Artificial_Intelligence_semantic.txt,26063,54,25960,50,1000,480.740741
2,Active_learning_(machine_learning).txt,Active_learning_(machine_learning)_semantic.txt,10291,18,10238,124,962,568.777778
3,Agent_harness.txt,Agent_harness_semantic.txt,5017,8,5000,131,905,625.000000
4,AI_art.txt,AI_art_semantic.txt,22332,42,22250,79,991,529.761905


In [98]:
print(
    "Documents processed:",
    len(semantic_summary_df)
)

print(
    "Total semantic chunks:",
    semantic_summary_df[
        "chunk_count"
    ].sum()
)

print(
    "Average chunks/document:",
    semantic_summary_df[
        "chunk_count"
    ].mean()
)

print(
    "Maximum chunk length:",
    semantic_summary_df[
        "max_chunk_length"
    ].max()
)

# ========================================
# Save the chunks 
# ========================================
SUMMARY_PATH = (
    SEMANTIC_DIR /
    "semantic_chunk_summary.csv"
)

semantic_summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)

print(
    "Summary saved to:",
    SUMMARY_PATH.resolve()
)

# ============================================
# Validate Chunk File Count
# ============================================

semantic_files = sorted(
    SEMANTIC_DIR.glob("*_semantic.txt")
)

print(
    "Semantic chunk files:",
    len(semantic_files)
)

if len(semantic_files) != 128:
    raise ValueError(
        f"Expected 128 semantic chunk files, "
        f"found {len(semantic_files)}."
    )

print(
    "Semantic chunk file count validation PASSED."
)

Documents processed: 128
Total semantic chunks: 4693
Average chunks/document: 36.6640625
Maximum chunk length: 1000
Summary saved to: C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\chunks\semantic\semantic_chunk_summary.csv
Semantic chunk files: 128
Semantic chunk file count validation PASSED.


In [99]:
# ============================================
# Validate Empty Files
# ============================================

empty_semantic_files = []

for file_path in semantic_files:

    content = file_path.read_text(
        encoding="utf-8"
    ).strip()

    if not content:
        empty_semantic_files.append(
            file_path.name
        )

print(
    "Empty semantic chunk files:",
    len(empty_semantic_files)
)

if empty_semantic_files:
    print(empty_semantic_files)

    raise ValueError(
        "Empty semantic chunk files detected."
    )

print(
    "Empty-file validation PASSED."
)



Empty semantic chunk files: 0
Empty-file validation PASSED.


In [100]:
documents_without_chunks = []

for file_path in semantic_files:

    content = file_path.read_text(
        encoding="utf-8"
    ).strip()

    chunks = [
        chunk.strip()
        for chunk in content.split("\n\n")
        if chunk.strip()
    ]

    if len(chunks) == 0:
        documents_without_chunks.append(
            file_path.name
        )

print(
    "Documents without chunks:",
    len(documents_without_chunks)
)

if documents_without_chunks:
    raise ValueError(
        "Some documents have no semantic chunks."
    )

print(
    "Document chunk-presence validation PASSED."
)

Documents without chunks: 0
Document chunk-presence validation PASSED.


In [101]:
oversized_chunks = []

for file_path in semantic_files:

    content = file_path.read_text(
        encoding="utf-8"
    ).strip()

    chunks = [
        chunk.strip()
        for chunk in content.split("\n\n")
        if chunk.strip()
    ]

    for index, chunk in enumerate(chunks):

        if len(chunk) > CHUNK_SIZE:

            oversized_chunks.append({
                "file": file_path.name,
                "chunk_index": index,
                "length": len(chunk),
                "preview": chunk[:500]
            })

oversized_df = pd.DataFrame(
    oversized_chunks
)

print(
    "Oversized chunks:",
    len(oversized_df)
)

display(
    oversized_df.head(10)
)

Oversized chunks: 0


""


In [102]:
short_chunks = []

for file_path in semantic_files:

    content = file_path.read_text(
        encoding="utf-8"
    ).strip()

    chunks = [
        chunk.strip()
        for chunk in content.split("\n\n")
        if chunk.strip()
    ]

    for index, chunk in enumerate(chunks):

        if len(chunk) < 50:

            short_chunks.append({
                "file": file_path.name,
                "chunk_index": index,
                "length": len(chunk),
                "content": chunk
            })

short_df = pd.DataFrame(
    short_chunks
)

print(
    "Chunks shorter than 50 characters:",
    len(short_df)
)

display(
    short_df.head(20)
)

Chunks shorter than 50 characters: 0


""


In [103]:
print("=" * 70)
print("SHORT CHUNK QUALITY AUDIT")
print("=" * 70)

print(f"Total short chunks (<50): {len(short_df)}")

display(
    short_df[
        ["file", "chunk_index", "length", "content"]
    ].head(50)
)

short_df["is_heading"] = short_df["content"].str.match(
    r"^==+.*==+$"
)

print("Heading-only chunks :", short_df["is_heading"].sum())
print("Other short chunks  :", (~short_df["is_heading"]).sum())

SHORT CHUNK QUALITY AUDIT
Total short chunks (<50): 0


KeyError: "None of [Index(['file', 'chunk_index', 'length', 'content'], dtype='str')] are in the [columns]"

In [ ]:
# ============================================================
# FINAL SEMANTIC CHUNKING VALIDATION
# ============================================================

total_documents = len(semantic_files)

total_chunks = 0
total_characters = 0

max_chunk_length = 0
min_chunk_length = None

oversized_chunks = []
short_chunks = []

for file_path in semantic_files:

    content = file_path.read_text(
        encoding="utf-8"
    ).strip()

    chunks = [
        chunk.strip()
        for chunk in content.split("\n\n")
        if chunk.strip()
    ]

    total_chunks += len(chunks)

    for index, chunk in enumerate(chunks):

        length = len(chunk)

        total_characters += length

        max_chunk_length = max(
            max_chunk_length,
            length
        )

        if min_chunk_length is None:
            min_chunk_length = length
        else:
            min_chunk_length = min(
                min_chunk_length,
                length
            )

        # ----------------------------------------------------
        # Oversized chunks
        # ----------------------------------------------------

        if length > CHUNK_SIZE:

            oversized_chunks.append({
                "file": file_path.name,
                "chunk_index": index,
                "length": length,
                "preview": chunk[:300]
            })

        # ----------------------------------------------------
        # Short chunks
        # ----------------------------------------------------

        if length < 50:

            short_chunks.append({
                "file": file_path.name,
                "chunk_index": index,
                "length": length,
                "content": chunk
            })


oversized_df = pd.DataFrame(
    oversized_chunks
)

short_df = pd.DataFrame(
    short_chunks
)

print("=" * 70)
print(
    "PHASE 4 – SEMANTIC CHUNKING VALIDATION"
)
print("=" * 70)

print(
    f"Documents processed       : {total_documents}"
)

print(
    f"Total semantic chunks     : {total_chunks}"
)

print(
    f"Total chunk characters    : "
    f"{total_characters:,}"
)

print(
    f"Minimum chunk length      : "
    f"{min_chunk_length}"
)

print(
    f"Maximum chunk length      : "
    f"{max_chunk_length}"
)

print(
    f"Maximum allowed length    : "
    f"{CHUNK_SIZE}"
)

print(
    f"Oversized chunks          : "
    f"{len(oversized_df)}"
)

print(
    f"Chunks shorter than 50    : "
    f"{len(short_df)}"
)

print("=" * 70)

# ============================================================
# HARD VALIDATION
# ============================================================

assert total_documents == 128, (
    f"Expected 128 documents, "
    f"found {total_documents}."
)

assert total_chunks > 0, (
    "No semantic chunks were generated."
)

assert min_chunk_length >= 50, (
    f"Minimum chunk length is {min_chunk_length}. "
    f"Expected at least 50 characters."
)

assert max_chunk_length <= CHUNK_SIZE, (
    f"Maximum chunk length is {max_chunk_length}. "
    f"Maximum allowed is {CHUNK_SIZE}."
)

assert len(oversized_df) == 0, (
    f"{len(oversized_df)} chunks exceed "
    f"the maximum size of {CHUNK_SIZE}."
)

assert len(short_df) == 0, (
    f"{len(short_df)} chunks are shorter than "
    f"the minimum allowed length of 50 characters."
)

print(
    "PHASE 4 SEMANTIC CHUNKING VALIDATION PASSED"
)

print("=" * 70)

PHASE 4 – SEMANTIC CHUNKING VALIDATION
Documents processed       : 128
Total semantic chunks     : 4693
Total chunk characters    : 2,353,687
Minimum chunk length      : 50
Maximum chunk length      : 1000
Maximum allowed length    : 1000
Oversized chunks          : 0
Chunks shorter than 50    : 0
PHASE 4 SEMANTIC CHUNKING VALIDATION PASSED
